In [ ]:
from pathlib import Path

import numpy as np
import soundfile as sf
from IPython.display import Audio, display

In [ ]:
search_locations = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
audio_directories = []
for location in search_locations:
    audio_directories.extend(
        [
            location / "ai-service" / "data" / "voice",
            location / "data" / "voice",
            location / "data" / "audio",
        ]
    )

audio_directory = next(
    (directory for directory in audio_directories if directory.is_dir()),
    None,
)
if audio_directory is None:
    raise FileNotFoundError("Could not locate an existing project audio directory.")

wav_files = sorted(
    (path for path in audio_directory.rglob("*.wav") if path.is_file()),
    key=lambda path: str(path).casefold(),
)
print(f"Audio directory: {audio_directory}")
print(f"Existing WAV files found: {len(wav_files)}")
if not wav_files:
    raise FileNotFoundError(f"No WAV recordings found in {audio_directory}")

In [ ]:
selected_wav = wav_files[0]
print(f"Selected existing recording: {selected_wav.name}")

In [ ]:
# Read the recording without changing the source file.
audio, sample_rate = sf.read(selected_wav, dtype="float32", always_2d=False)
audio_shape = audio.shape
channels = 1 if audio.ndim == 1 else audio.shape[1]
frame_count = audio.shape[0]
duration_seconds = frame_count / sample_rate if sample_rate > 0 else 0.0

In [ ]:
print(f"Filename: {selected_wav.name}")
print(f"Sample rate: {sample_rate} Hz")
print(f"Duration: {duration_seconds:.2f} seconds")
print(f"Channels: {channels}")
print(f"Audio shape: {audio_shape}")

In [ ]:
display(Audio(data=audio, rate=sample_rate))

In [ ]:
validation_results = {
    "selected file exists": selected_wav.is_file(),
    "selected file is inside the audio directory": selected_wav.is_relative_to(audio_directory),
    "WAV extension": selected_wav.suffix.casefold() == ".wav",
    "sample rate is positive": sample_rate > 0,
    "audio has at least one frame": frame_count > 0,
    "audio has one or more channels": channels > 0,
    "audio samples are finite": bool(np.isfinite(audio).all()),
    "duration is positive": duration_seconds > 0,
}
for check, passed in validation_results.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check}")

all_checks_passed = all(validation_results.values())
print(f"\nFINAL RESULT: {'PASS' if all_checks_passed else 'FAIL'}")
assert all_checks_passed, "Audio validation failed; review the checks above."